# Tutorial 04 — Audit Trail

**No API key required. Fully deterministic.**

Every tool result in eXo-brain carries a **correlation ID**. This tutorial shows how to
emit audit records into an audit store using that correlation ID.
This tutorial shows how to:
- Wire the audit pipeline (store + pipeline + logger)
- Execute a tool and capture its audit correlation ID
- Query audit records by correlation ID
- Build and verify a SHA-256 hash chain
- Prove tamper-evidence by mutating a record
- Compute the chain fingerprint with `compute_audit_chain_fingerprint`

This is the foundation of compliance reporting and SOC 2 evidence. Signed/sealed bundles
require an external anchoring step (signature, append-only store, or sealed fingerprint),
which is described elsewhere in the repo.

In [1]:
import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))
try:
    from dotenv import load_dotenv
    load_dotenv(_root / ".env", override=False)
except ImportError:
    pass

## Part 1 — Wire the audit infrastructure

Three components work together:
- `InMemoryAuditStore` — persists `AuditRecord` objects, queryable by correlation ID
- `ToolAuditPipeline` — emits structured audit events into the store via async `emit()`
- `StructuredLogger` — records every emit as a structured log entry (in-memory by default)

In [2]:
from src.audit.trail import AuditChainRecord, chain_record, verify_chain
from src.persistence.audit_store import InMemoryAuditStore
from src.persistence.contracts import AuditRecord
from src.observability.tool_audit import ToolAuditPipeline
from src.observability.logging import StructuredLogger, LogLevel
from src.compliance.evidence_bundle import compute_audit_chain_fingerprint
from src.tools.executor import DeterministicToolExecutor
from src.tools.registry import ToolRegistry, ToolDescriptor
from src.schemas.tool_io import (
    RiskTier, ToolCallContext, ToolStatus, ToolExecutionMode,
)
from src.policies.middleware import DeterministicFirstPolicyMiddleware

# Wire audit infrastructure
audit_store = InMemoryAuditStore()
logger = StructuredLogger()
audit_pipeline = ToolAuditPipeline(logger=logger, audit_store=audit_store)

print("audit_store  :", type(audit_store).__name__)
print("logger       :", type(logger).__name__)
print("audit_pipeline:", type(audit_pipeline).__name__)

audit_store  : InMemoryAuditStore
logger       : StructuredLogger
audit_pipeline: ToolAuditPipeline


## Part 2 — Execute a tool and capture the correlation ID

We register a simple tool, wire `DeterministicToolExecutor` with policy middleware,
and execute one call. The executor sets `ToolResult.audit.correlation_id` on every result.

In [3]:
# Register a simple tool
registry = ToolRegistry()
registry.register(ToolDescriptor(
    name="add_numbers",
    handler=lambda a, b: {"sum": a + b},
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
    description="Returns the sum of two numbers.",
))

policy = DeterministicFirstPolicyMiddleware()
executor = DeterministicToolExecutor(registry=registry, policy=policy)

# Build a ToolCallContext — schema_version and all ID fields are required
call = ToolCallContext(
    schema_version="1.0",
    call_id="call-audit-demo-001",
    session_id="session-audit-001",
    run_id="run-audit-001",
    job_id="job-audit-001",
    task_id="task-audit-001",
    agent_id="agent-audit-001",
    provider_id="demo",
    tool_name="add_numbers",
    arguments={"a": 7, "b": 3},
    tenant_id="tenant-acme",
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
)

result = executor.execute(call)

print("status           :", result.status)
print("result           :", result.result)
print("audit.correlation_id:", result.audit.correlation_id if result.audit else "MISSING")

correlation_id = result.audit.correlation_id

status           : ToolStatus.SUCCESS
result           : {'value': {'sum': 10}}
audit.correlation_id: call-audit-demo-001


## Part 3 — Emit an audit event and query the store

`ToolAuditPipeline.emit()` is async. It appends an `AuditRecord` to the store and logs it.
We then query back by correlation ID to inspect the full record.

In [4]:
import asyncio
try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

async def emit_and_query():
    # Emit a tool.executed event linked to our correlation ID
    await audit_pipeline.emit(
        event_type="tool.executed",
        correlation_id=correlation_id,
        tenant_id="tenant-acme",
        payload={
            "tool_name": "add_numbers",
            "status": result.status.value,
            "result": result.result,
        },
    )

    # Query back by correlation ID
    records = await audit_store.query_audit_events(
        correlation_id=correlation_id,
        tenant_id="tenant-acme",
    )
    return records

audit_records = asyncio.run(emit_and_query())

print(f"Records found: {len(audit_records)}")
for r in audit_records:
    print()
    print("  event_id      :", r.event_id)
    print("  correlation_id:", r.correlation_id)
    print("  tenant_id     :", r.tenant_id)
    print("  event_type    :", r.event_type)
    print("  payload       :", r.payload)

Records found: 1

  event_id      : audit_952fdd98772a
  correlation_id: call-audit-demo-001
  tenant_id     : tenant-acme
  event_type    : tool.executed
  payload       : {'tool_name': 'add_numbers', 'status': 'success', 'result': {'value': {'sum': 10}}}


## Part 4 — Build a SHA-256 hash chain manually

`chain_record(payload, previous_hash)` computes `SHA-256(json(payload) + previous_hash)`.
The chain starts with `previous_hash = ""` (genesis record).
Each record links to the previous via its hash — making post-hoc alteration **detectable** during
verification, especially when checked against stored hashes or an anchored fingerprint.

In [5]:
# Build three audit events as plain dicts
event_payloads = [
    {"event_type": "session.started",   "tenant_id": "tenant-acme", "correlation_id": "corr-001"},
    {"event_type": "tool.executed",     "tenant_id": "tenant-acme", "tool_name": "add_numbers", "status": "success"},
    {"event_type": "session.completed", "tenant_id": "tenant-acme", "correlation_id": "corr-001"},
]

# Build the chain — genesis record uses previous_hash=""
chain: list[AuditChainRecord] = []
prev_hash = ""
for payload in event_payloads:
    record = chain_record(payload, prev_hash)
    chain.append(record)
    prev_hash = record.record_hash

print("Chain records:")
for i, r in enumerate(chain):
    print(f"  [{i}] prev_hash  : {r.previous_hash[:16] or '(genesis)':>16}...")
    print(f"      record_hash: {r.record_hash[:16]}...")
    print(f"      payload    : {r.payload['event_type']}")
    print()

Chain records:
  [0] prev_hash  :        (genesis)...
      record_hash: 702453dc52fd1ab4...
      payload    : session.started

  [1] prev_hash  : 702453dc52fd1ab4...
      record_hash: 7a3582ff2c8e5675...
      payload    : tool.executed

  [2] prev_hash  : 7a3582ff2c8e5675...
      record_hash: fa769a3ebb877cd5...
      payload    : session.completed



## Part 5 — Verify the chain

`verify_chain(records)` recomputes every hash and checks linkage.
Returns `True` when the chain is intact.

In [6]:
is_valid = verify_chain(chain)
print(f"Chain valid (unmodified): {is_valid}")
assert is_valid, "Chain should be valid before any mutation"
print("PASS — chain integrity confirmed")

Chain valid (unmodified): True
PASS — chain integrity confirmed


## Part 6 — Prove tamper-evidence

If any record's payload is modified after the chain is built, `verify_chain` detects the break.
The hash recomputed for the mutated record will not match the stored `record_hash`.

In [7]:
import copy

# Deep-copy so we keep the original intact
tampered_chain = copy.deepcopy(chain)

# Silently mutate the middle record's payload
tampered_chain[1].payload["status"] = "success_FORGED"

is_still_valid = verify_chain(tampered_chain)
print(f"Chain valid after mutation: {is_still_valid}")
assert not is_still_valid, "Mutated chain must fail verification"
print("PASS — tamper-evidence works: mutation detected by hash chain")

# Original chain is untouched
assert verify_chain(chain), "Original chain must still be valid"
print("PASS — original chain still intact")

Chain valid after mutation: False
PASS — tamper-evidence works: mutation detected by hash chain
PASS — original chain still intact


## Part 7 — Compute the chain fingerprint

`compute_audit_chain_fingerprint` takes a list of plain dicts (the serialised form of records)
and returns `(chain_valid: bool, last_hash: str)`.

This is what the audit export API uses as the **fingerprint input** for a signed/sealed
bundle. Signing/anchoring is an additional step not demonstrated in this notebook.

In [8]:
# Serialize chain records to plain dicts (as the API export layer does)
records_as_dicts = [
    {
        "payload":       r.payload,
        "previous_hash": r.previous_hash,
        "record_hash":   r.record_hash,
    }
    for r in chain
]

chain_valid, last_hash = compute_audit_chain_fingerprint(records_as_dicts)

print(f"chain_valid : {chain_valid}")
print(f"last_hash   : {last_hash[:32]}...")
assert chain_valid, "Fingerprint must confirm chain is valid"
print()
print("PASS — compute_audit_chain_fingerprint returned (True, <hash>)")

chain_valid : True
last_hash   : c8d5b869911cc1743ad8c68b1f6546b2...

PASS — compute_audit_chain_fingerprint returned (True, <hash>)


## Summary

| Capability | Module | Key function / class |
|---|---|---|
| Structured audit emit | `src/observability/tool_audit` | `ToolAuditPipeline.emit()` |
| In-memory audit persistence | `src/persistence/audit_store` | `InMemoryAuditStore` |
| SHA-256 hash chain | `src/audit/trail` | `chain_record`, `verify_chain` |
| Tamper detection | `src/audit/trail` | `verify_chain` → `False` on mutation |
| Fingerprint for export | `src/compliance/evidence_bundle` | `compute_audit_chain_fingerprint` |
| Correlation-linked tool result | `src/tools/executor` | `ToolResult.audit.correlation_id` |

**Key insight:** Every tool result carries a correlation ID, and you can emit correlation-linked
audit records into your store/pipeline using that ID. The SHA-256 hash chain makes post-hoc
mutation **detectable** when the chain is verified against stored hashes or an externally
anchored fingerprint — any record edit breaks `verify_chain`.

### Next steps
- **Tutorial 05** — Multi-turn sessions: how session state, timeline, and quota thread across turns
- **Tutorial 06** — Background workflows: DAG execution, retries, and checkpoint-based resume

## Notebook navigation

| If you want… | Open |
|---|---|
| Previous / next in learning path | See `notebooks/README.md` index |
| Fast module smoke after a code change | `check_01` … `check_04` |
| Ingress or tool boundary proofs | `edge_01`, `edge_02` |
| Full governance lab (story + optional live) | `tutorial_08_governed_execution_sandbox.ipynb` |
| Evaluator time-boxed paths | `notebooks/EVALUATOR_GUIDE.md` |

**Regenerate notebooks:** edit this build script, then `python notebooks/build_tutorials.py` (do not hand-edit `.ipynb` JSON).